In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, losses, metrics

In [2]:
# ==========================================
# 1. SYNTHETIC MULTIMODAL DATA GENERATOR
# ==========================================
def generate_synthetic_data(num_samples=600, seq_len=30, num_telemetry_feats=4, img_size=(224, 224)):
    """
    Simulates synchronized thermal IR images and battery telemetry sequences:
    - 1D Telemetry: Voltage (V), Current (I), Temperature (T), Internal Resistance (R_int)
    - 2D Thermal: RGB simulated surface temperature matrices
    - Targets: 
        1. SOH (Regression): Float between 0.0 and 1.0 (Capacity fade)
        2. Runaway Risk (Classification): 0: Safe, 1: Warning, 2: Critical
    """
    print(f"Generating {num_samples} multimodal samples...")

    # 1D Telemetry Sequences: (N, seq_len, num_telemetry_feats)
    X_telemetry = np.random.uniform(0.0, 1.0, size=(num_samples, seq_len, num_telemetry_feats)).astype(np.float32)

    # 2D Infrared Frames: (N, 224, 224, 3) normalized to [0, 1]
    X_images = np.random.uniform(0.0, 1.0, size=(num_samples, img_size[0], img_size[1], 3)).astype(np.float32)

    # SOH Target (Continuous fade: 1.0 down to ~0.6)
    y_soh = np.random.uniform(0.60, 1.00, size=(num_samples, 1)).astype(np.float32)

    # Risk Target (3 classes: 0=Safe, 1=Warning, 2=Critical)
    # Higher temperature/resistance correlates with higher risk
    risk_labels = np.random.choice([0, 1, 2], size=num_samples, p=[0.60, 0.25, 0.15])
    y_risk = tf.keras.utils.to_categorical(risk_labels, num_classes=3)

    return (X_images, X_telemetry), (y_soh, y_risk)

In [3]:
# 2. DUAL-BRANCH MULTIMODAL ARCHITECTURE
# ==========================================
def build_multimodal_model(img_shape=(224, 224, 3), seq_shape=(30, 4), num_classes=3):
    # -----------------------------
    # Branch A: Spatial (Thermal IR)
    # -----------------------------
    image_input = layers.Input(shape=img_shape, name="thermal_image_input")
    
    # Pre-trained MobileNetV2 backbone (feature extraction)
    base_mobilenet = tf.keras.applications.MobileNetV2(
        input_shape=img_shape,
        include_top=False,
        weights="imagenet"
    )
    base_mobilenet.trainable = False  # Freeze base weights for fast convergence
    
    spatial_features = base_mobilenet(image_input)
    spatial_pooled = layers.GlobalAveragePooling2D(name="spatial_gap")(spatial_features)
    spatial_dense = layers.Dense(128, activation="relu", name="spatial_embedding")(spatial_pooled)
    spatial_out = layers.Dropout(0.2)(spatial_dense)

    # -----------------------------
    # Branch B: Temporal (1D Telemetry)
    # -----------------------------
    telemetry_input = layers.Input(shape=seq_shape, name="telemetry_seq_input")
    
    temporal_conv = layers.Conv1D(filters=32, kernel_size=3, padding="same", activation="relu")(telemetry_input)
    temporal_lstm = layers.Bidirectional(layers.LSTM(64, return_sequences=False), name="bilstm_layer")(temporal_conv)
    temporal_dense = layers.Dense(64, activation="relu", name="temporal_embedding")(temporal_lstm)
    temporal_out = layers.Dropout(0.2)(temporal_dense)

    # -----------------------------
    # Attention-Guided Late Fusion
    # -----------------------------
    fused_features = layers.concatenate([spatial_out, temporal_out], name="feature_concatenation")
    
    # Self-attention gating layer to dynamically weight modalities
    attention_weights = layers.Dense(192, activation="softmax", name="fusion_attention_weights")(fused_features)
    gated_features = layers.multiply([fused_features, attention_weights], name="attention_multiplication")
    
    fusion_dense = layers.Dense(64, activation="relu", name="shared_representation")(gated_features)

    # -----------------------------
    # Multi-Task Output Heads
    # -----------------------------
    # Head 1: Continuous SOH % estimation
    soh_output = layers.Dense(1, activation="linear", name="soh_output")(fusion_dense)

    # Head 2: Thermal Runaway Risk state classification
    risk_output = layers.Dense(num_classes, activation="softmax", name="risk_output")(fusion_dense)

    # Compile unified Functional model
    model = models.Model(
        inputs=[image_input, telemetry_input],
        outputs=[soh_output, risk_output],
        name="Multimodal_EV_Battery_Health_Net"
    )

    return model

In [6]:
# 3. PIPELINE EXECUTION & TRAINING
# ==========================================
if __name__ == "__main__":
    # Create directory for model artifacts
    os.makedirs("models", exist_ok=True)

    # 1. Prepare Data
    (X_img_train, X_tel_train), (y_soh_train, y_risk_train) = generate_synthetic_data(num_samples=500)
    (X_img_val, X_tel_val), (y_soh_val, y_risk_val) = generate_synthetic_data(num_samples=100)

    # 2. Build Model
    model = build_multimodal_model()
    model.summary()

    # 3. Configure Multi-Task Losses and Metrics
    model.compile(
        optimizer=optimizers.Adam(learning_rate=1e-3),
        loss={
            "soh_output": losses.MeanSquaredError(),
            "risk_output": losses.CategoricalCrossentropy()
        },
        loss_weights={
            "soh_output": 1.0,   # Regression scale
            "risk_output": 2.0   # Emphasize safety-critical classification
        },
        metrics={
            "soh_output": [metrics.MeanAbsoluteError(name="mae"), metrics.RootMeanSquaredError(name="rmse")],
            "risk_output": [metrics.CategoricalAccuracy(name="accuracy")]
        }
    )

    # 4. Train
    print("\nStarting Training Pipeline...")
    history = model.fit(
        x={"thermal_image_input": X_img_train, "telemetry_seq_input": X_tel_train},
        y={"soh_output": y_soh_train, "risk_output": y_risk_train},
        validation_data=(
            {"thermal_image_input": X_img_val, "telemetry_seq_input": X_tel_val},
            {"soh_output": y_soh_val, "risk_output": y_risk_val}
        ),
        epochs=10,
        batch_size=16,
        verbose=1
    )

    # 5. Export Model Artifact
    export_path = os.path.join("models", "battery_multimodal_model.keras")
    model.save(export_path)
    print(f"\nModel successfully exported to: {export_path}")

Generating 500 multimodal samples...
Generating 100 multimodal samples...


Model: "Multimodal_EV_Battery_Health_Net"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ thermal_image_input           │ (None, 224, 224, 3)       │               0 │ -                          │
│ (InputLayer)                  │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ telemetry_seq_input           │ (None, 30, 4)             │               0 │ -                          │
│ (InputLayer)                  │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ mobilenetv2_1.00_224          │ (None, 7, 7, 1280)        │       2,257,984 │ thermal_image_input[0][0]  │
│ (Functional)                  │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv1d_1 (Conv1D)             │ (None, 30, 32)            │             416 │ telemetry_seq_input[0][0]  │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ spatial_gap                   │ (None, 1280)              │               0 │ mobilenetv2_1.00_224[0][0] │
│ (GlobalAveragePooling2D)      │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ bilstm_layer (Bidirectional)  │ (None, 128)               │          49,664 │ conv1d_1[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ spatial_embedding (Dense)     │ (None, 128)               │         163,968 │ spatial_gap[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ temporal_embedding (Dense)    │ (None, 64)                │           8,256 │ bilstm_layer[0][0]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout_2 (Dropout)           │ (None, 128)               │               0 │ spatial_embedding[0][0]    │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout_3 (Dropout)           │ (None, 64)                │               0 │ temporal_embedding[0][0]   │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ feature_concatenation         │ (None, 192)               │               0 │ dropout_2[0][0],           │
│ (Concatenate)                 │                           │                 │ dropout_3[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ fusion_attention_weights      │ (None, 192)               │          37,056 │ feature_concatenation[0][… │
│ (Dense)                       │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ attention_multiplication      │ (None, 192)               │               0 │ feature_concatenation[0][… │
│ (Multiply)                    │                           │                 │ fusion_attention_weights[… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ shared_representation (Dense) │ (None, 64)                │          12,35

 Total params: 2,529,956 (9.65 MB)

 Trainable params: 271,972 (1.04 MB)

 Non-trainable params: 2,257,984 (8.61 MB)


Starting Training Pipeline...
Epoch 1/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 21s 335ms/step - loss: 2.1978 - risk_output_accuracy: 0.6200 - risk_output_loss: 0.9927 - soh_output_loss: 0.2018 - soh_output_mae: 0.3467 - soh_output_rmse: 0.4528 - val_loss: 1.7376 - val_risk_output_accuracy: 0.6900 - val_risk_output_loss: 0.8333 - val_soh_output_loss: 0.0132 - val_soh_output_mae: 0.0971 - val_soh_output_rmse: 0.1140
Epoch 2/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 9s 268ms/step - loss: 2.0179 - risk_output_accuracy: 0.6200 - risk_output_loss: 0.9711 - soh_output_loss: 0.0840 - soh_output_mae: 0.2125 - soh_output_rmse: 0.2912 - val_loss: 1.6888 - val_risk_output_accuracy: 0.6900 - val_risk_output_loss: 0.8008 - val_soh_output_loss: 0.0148 - val_soh_output_mae: 0.1021 - val_soh_output_rmse: 0.1255
Epoch 3/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 9s 275ms/step - loss: 1.9442 - risk_output_accuracy: 0.6200 - risk_output_loss: 0.9298 - soh_output_loss: 0.0658 - soh_output_mae: 0.1923 - soh_output_rmse: 0.2567 - val_loss: 1.